In [21]:
import os
import time
import tiktoken
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, hamming_loss


from datasets import load_dataset

In [22]:
ds = load_dataset("TimSchopf/arxiv_categories", "default")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20397 entries, 0 to 20396
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype              
---  ------         --------------  -----              
 0   id             20397 non-null  object             
 1   title          20397 non-null  object             
 2   abstract       20397 non-null  object             
 3   categories     20397 non-null  object             
 4   creation_date  20397 non-null  datetime64[ns, UTC]
dtypes: datetime64[ns, UTC](1), object(4)
memory usage: 796.9+ KB


In [23]:
test = test.rename(columns={'title': 'text'})
test = test.rename(columns={'categories': 'labels'})

In [24]:
def clean_element(lst):
    final = []
    for elem in lst:
        clean = elem.split('->')[-1]
        final.append(clean)
    return final

test['labels'] = test['labels'].apply(clean_element)

test

,id,text,abstract,labels,creation_date
0,2208.11718,gSwin: Gated MLP Vision Model with Hierarchica...,"Following the success in language domain, the ...","[cs.CV, cs.LG]",2022-08-24 18:00:46+00:00
1,2302.13571,FLAG: Fast Label-Adaptive Aggregation for Mult...,Federated learning aims to share private data ...,"[cs.AI, cs.LG]",2023-02-27 08:16:39+00:00
2,2208.12842,Advances in device-independent quantum key dis...,Device-independent quantum key distribution (D...,[quant-ph],2022-08-26 18:55:40+00:00
3,2303.07834,Finite-Horizon Constrained MDPs With Both Addi...,This paper considers the problem of finding a ...,"[math.OC, math.PR]",2023-03-14 12:15:58+00:00
4,1504.07490,Exact equations for structure functions and eq...,We derive equations for the source terms appea...,[physics.flu-dyn],2015-04-28 14:17:44+00:00
...,...,...,...,...,...
20392,1608.05170,Resource efficient redundancy using quorum-bas...,In this paper we propose a cycle redundancy te...,[cs.NI],2016-08-18 04:35:50+00:00
20393,cond-mat/9504003,An almost sure large deviation principle for t...,We prove a large deviation principle for the f...,[cond-mat],1995-04-03 11:11:22+00:00
20394,1412.5555,Local stability of Kolmogorov forward equation...,The focus of this work is on local stability o...,[math.PR],2014-12-17 20:03:13+00:00
20395,astro-ph/0408428,An Analysis of the Large Scale N-body Simulati...,We analyze the Minkowski functionals with a la...,[astro-ph],2004-08-24 06:12:43+00:00


In [25]:
labels = ["cs.AI", "cs.CL", "stat.ML", "math.OC", "cs.LG"]

test = test[test['labels'].apply(lambda cats: all(c in labels for c in cats))]
test = test[test['labels'].apply(len) > 0]
test.drop(columns=['id','abstract','creation_date'], inplace=True)
test.reset_index(drop=True, inplace=True)

test

,text,labels
0,FLAG: Fast Label-Adaptive Aggregation for Mult...,"[cs.AI, cs.LG]"
1,Large-Margin Classification in Hyperbolic Space,"[cs.LG, stat.ML]"
2,Student Specialization in Deep ReLU Networks W...,"[cs.LG, stat.ML]"
3,Joint Modelling of Emotion and Abusive Languag...,"[cs.CL, cs.LG]"
4,Allocation of Excitation Signals for Generic I...,[math.OC]
...,...,...
1265,Benign Overfitting in Two-layer Convolutional ...,"[cs.LG, math.OC, stat.ML]"
1266,An FPGA-Based On-Device Reinforcement Learning...,"[cs.LG, stat.ML]"
1267,Something for (almost) nothing: Improving deep...,[cs.LG]
1268,An Alternative to Variance: Gini Deviation for...,"[cs.AI, cs.LG]"


In [26]:
mlb = MultiLabelBinarizer()
test_labels_binarized = mlb.fit_transform(test['labels'])

test_labels_df = pd.DataFrame(test_labels_binarized, columns=mlb.classes_)

test = pd.concat([test, test_labels_df], axis=1)

test.drop(columns=['labels'], inplace=True)

test

,text,cs.AI,cs.CL,cs.LG,math.OC,stat.ML
0,FLAG: Fast Label-Adaptive Aggregation for Mult...,1,0,1,0,0
1,Large-Margin Classification in Hyperbolic Space,0,0,1,0,1
2,Student Specialization in Deep ReLU Networks W...,0,0,1,0,1
3,Joint Modelling of Emotion and Abusive Languag...,0,1,1,0,0
4,Allocation of Excitation Signals for Generic I...,0,0,0,1,0
...,...,...,...,...,...,...
1265,Benign Overfitting in Two-layer Convolutional ...,0,0,1,1,1
1266,An FPGA-Based On-Device Reinforcement Learning...,0,0,1,0,1
1267,Something for (almost) nothing: Improving deep...,0,0,1,0,0
1268,An Alternative to Variance: Gini Deviation for...,1,0,1,0,0


In [27]:
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [28]:
def classify(text, labels):
    start_time = time.time()

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        store=True,
        messages = [
            {"role": "system", "content": "You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling multilabel classification tasks based on user instructions."},
            {"role": "user", "content": f"Classify the following text based on the task: Classification of computer science-based arXiv articles. Only respond with the labels that best describe the text. The possible labels are: {', '.join(labels)}. Text: {text}"}
        ],
    )

    request_time = time.time() - start_time
    completion = response.choices[0].message.content.lower()
    completion_tokens = response.usage.completion_tokens
    prompt_tokens = response.usage.prompt_tokens
    total_tokens = response.usage.total_tokens

    return completion, request_time, completion_tokens, prompt_tokens, total_tokens

def post_process(text):
    list = []
    if 'artificial intelligence' in text:
        list.append('cs.AI')
    if 'computation and language' in text:
        list.append('cs.CL')
    if 'statistics machine learning' in text:
        list.append('stat.ML')
    if 'optimization and control' in text:
        list.append('math.OC')
    if 'computational machine learning' in text:
        list.append('cs.LG')

    return list

In [29]:
pred_df = test.copy() 
labels_for_prompt = ["Artificial Intelligence", "Computation and Language", "Statistics Machine Learning", "Optimization and Control", "Computational Machine Learning"]

for index, row in pred_df.iterrows():
    try:
        text = row['text']
        completion, request_time, completion_tokens, prompt_tokens, total_tokens = classify(text, labels_for_prompt)
        pred_df.at[index, 'prediction'] = completion
        pred_df.at[index, 'request_time'] = request_time
        pred_df.at[index, 'completion_tokens'] = completion_tokens
        pred_df.at[index, 'prompt_tokens'] = prompt_tokens
        pred_df.at[index, 'total_tokens'] = total_tokens

    except Exception as e:
        # Save the current state of the DataFrame to a file before breaking out or retrying.
        pred_df.to_csv("results/partial_openai_ZS_multilabel3.csv", index=False)
        print(f"An error occurred at index {index}: {e}. Partial results saved.")
        # Optionally, you can break out of the loop or continue based on your needs.
        break

pred_df['prediction_post_processed'] = pred_df['prediction'].apply(post_process)
pred_df.to_csv("results/openai_ZS_multilabel3.csv", index=False)

pred_df

KeyboardInterrupt: 

In [ ]:
for label in labels:
    pred_df[f"{label} pred"] = pred_df.apply(lambda row: 1 if label in row['prediction_post_processed'] else 0, axis=1)

pred_df = pred_df.drop(columns=['prediction'])

pred_df

,text,cs.AI,cs.CL,cs.LG,math.OC,stat.ML,request_time,completion_tokens,prompt_tokens,total_tokens,prediction_post_processed,cs.AI pred,cs.CL pred,stat.ML pred,math.OC pred,cs.LG pred
0,FLAG: Fast Label-Adaptive Aggregation for Mult...,1,0,1,0,0,1.078638,12.0,123.0,135.0,[],0,0,0,0,0
1,Large-Margin Classification in Hyperbolic Space,0,0,1,0,1,0.424648,8.0,116.0,124.0,[],0,0,0,0,0
2,Student Specialization in Deep ReLU Networks W...,0,0,1,0,1,0.487548,8.0,122.0,130.0,[],0,0,0,0,0
3,Joint Modelling of Emotion and Abusive Languag...,0,1,1,0,0,0.591278,8.0,117.0,125.0,[],0,0,0,0,0
4,Allocation of Excitation Signals for Generic I...,0,0,0,1,0,0.556943,4.0,121.0,125.0,[],0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1265,Benign Overfitting in Two-layer Convolutional ...,0,0,1,1,1,0.659561,8.0,120.0,128.0,[],0,0,0,0,0
1266,An FPGA-Based On-Device Reinforcement Learning...,0,0,1,0,1,0.578767,8.0,121.0,129.0,[],0,0,0,0,0
1267,Something for (almost) nothing: Improving deep...,0,0,1,0,0,0.587865,12.0,122.0,134.0,[],0,0,0,0,0
1268,An Alternative to Variance: Gini Deviation for...,1,0,1,0,0,0.889224,12.0,123.0,135.0,[],0,0,0,0,0


In [ ]:
y_true = pred_df[labels].values
y_pred = pred_df[[f"{label} pred" for label in labels]].values

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)
hamming_loss = hamming_loss(y_true, y_pred)
print('Hamming loss: %f' % hamming_loss)

Accuracy: 0.000000
F1 score: 0.000000
Precision: 0.000000
Recall: 0.000000
Hamming loss: 0.304567


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# get average response time, vram usage and ram usage
request_time_avg = pred_df['request_time'].mean()
completion_tokens_avg = pred_df['completion_tokens'].mean()
prompt_tokens_avg = pred_df['prompt_tokens'].mean()
total_tokens_avg = pred_df['total_tokens'].mean()

print(f'Average response time: {request_time_avg}')
print(f'Average completion tokens: {completion_tokens_avg}')
print(f'Average prompt tokens: {prompt_tokens_avg}')
print(f'Average total tokens: {total_tokens_avg}')

Average response time: 0.650947471303264
Average completion tokens: 7.939370078740158
Average prompt tokens: 120.22204724409448
Average total tokens: 128.16141732283464


In [ ]:
input_token_price = 0.15/1_000_000
output_token_price = 0.6/1_000_000

def count_tokens(text, model="gpt-4o-mini"):
    try:
        # Try to get the encoding for the given model
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        # If the model isn't recognized, fall back to a default encoding
        encoding = tiktoken.get_encoding("cl100k_base")
    
    tokens = encoding.encode(text)
    return len(tokens)

# Calculate the cost of the requests
total_cost = 0
for index, row in pred_df.iterrows():
    completion_tokens = row['completion_tokens']
    prompt_tokens = row['prompt_tokens']
    cost  = completion_tokens * output_token_price + prompt_tokens * input_token_price
    total_cost += cost

print(f'Total cost: USD {total_cost}')

Total cost: USD 0.028952100000000015


In [ ]:
with open('results/openai_ZS_multilabel3.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Hamming loss: {hamming_loss}\n')
    f.write(f'Average response time: {request_time_avg}\n')
    f.write(f'Average completion tokens: {completion_tokens_avg}\n')
    f.write(f'Average prompt tokens: {prompt_tokens_avg}\n')
    f.write(f'Average total tokens: {total_tokens_avg}\n')
    f.write(f'Total cost: USD {total_cost}\n')